In [2]:
import pandas as pd

In [3]:
## Read the missing essential 61 fields for analysis
df_maven  = pd.read_csv('input/essential_61.csv',na_filter=False)
len(df_maven)

61

In [4]:
df_maven

,Priority,Model,Source Field Name,Source Label,CSTE Label
0,,General,BIRTH_DATE,Birth date,Birth Date
1,,Hepatitis,BIRTH_DATE,Birth date,Birth Date
2,,STD,BIRTH_DATE,Birth date,Birth Date
3,,General,CASE_INVESTIGATION_STATUS,Case investigation status,Case Class Status Code
4,,Hepatitis,CASE_INVESTIGATION_STATUS,,Case Class Status Code
...,...,...,...,...,...
56,,Childhood Lead,RACE_OTHER_SPECIFY,,Other Race Text
57,,STD,REPORT_DATE,,Date Reported
58,,General,REPORT_TO_CDC,Report case to CDC,Immediate National Notifiable Condition
59,,Hepatitis,REPORT_TO_CDC,Report Case to CDC,Immediate National Notifiable Condition


In [ ]:
# Create string compare statements for SQL query

for m,q in zip(df_maven.loc[:,'Model'],df_maven.loc[:,'Source Field Name']):
    new_m=''
    if m=='General':
        new_m = 'DiseaseSurveillanceModel_General'
    elif m=='STD':
         new_m='DiseaseSurveillanceModel_STD'
    elif m=='Hepatitis':
        new_m= 'DiseaseSurveillanceModel_Hep'
    elif m=='Childhood Lead':
        new_m = 'Childhood_Lead_Child_Model'
    elif m=='Coinfection':
        new_m = 'DiseaseSurveillanceModel_Coinfection'
    elif m=='Cluster':
        new_m = 'ClusterModel'
    print('('+"c.MODEL_NAME"+" like '"+new_m+"' and "+'a.QUESTION_ID'+" like '"+q+"%') or" )

(c.MODEL_NAME like 'DiseaseSurveillanceModel_General' and a.QUESTION_ID like 'BIRTH_DATE%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_Hep' and a.QUESTION_ID like 'BIRTH_DATE%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'BIRTH_DATE%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_General' and a.QUESTION_ID like 'CASE_INVESTIGATION_STATUS%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_Hep' and a.QUESTION_ID like 'CASE_INVESTIGATION_STATUS%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'CDR_SYMPTOMS%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'CDR_SYMPTOMS_CLINICIAN_OBS%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'CDR_SYMPTOMS_CLINICIAN_OBS_DATE%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'CDR_SYMPTOMS_PATIENT_DESCRIBED%') or
(c.MODEL_NAME like 'DiseaseSurveillanceModel_STD' and a.QUESTION_ID like 'CDR_SYMPTO

In [ ]:
# Used the output above to create a SQL query to get count values for those 61 elements

In [76]:
df_query = pd.read_csv('input/mvp_remapping_db_results.csv',na_filter=False)
len(df_query)

198

In [77]:
df_query

,count_of_values,value,question_id,model_name,case_count,fill_rate
0,131749,12/31/2011,BIRTH_DATE,STD,131753,99.996964
1,281304,12/31/2025,BIRTH_DATE,General,283121,99.358225
2,50137,12/31/2018,BIRTH_DATE,Hepatitis,50139,99.996011
3,46,12/17/2025,BIRTH_DATE_CPY,STD,131753,0.034914
4,352,ASSIGNED,CASE_INVESTIGATION_STATUS_ASSIGNED,General,283121,0.124328
...,...,...,...,...,...,...
193,29,Original interview - N/A - N/A,SYMPTOMS_INTERVIEW,STD,131753,0.022011
194,145,12/28/2025,SYMPTOMS_PATIENT_DESCRIBED_DATE,STD,131753,0.110054
195,6,NO,SYMPTOMS_PATIENT_DESCRIBED_NO,STD,131753,0.004554
196,2,UNKNOWN,SYMPTOMS_PATIENT_DESCRIBED_UNKNOWN,STD,131753,0.001518


In [78]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:]==answer):
        return question[:start].strip('_')
    else:
        return question

In [80]:
df_query.loc[:,'parent_question_id']  = [ join_all_except_last(q,a) for q,a in zip(df_query.loc[:,'question_id'],df_query.loc[:,'value'])]

In [81]:
df_query

,count_of_values,value,question_id,model_name,case_count,fill_rate,parent_question_id
0,131749,12/31/2011,BIRTH_DATE,STD,131753,99.996964,BIRTH_DATE
1,281304,12/31/2025,BIRTH_DATE,General,283121,99.358225,BIRTH_DATE
2,50137,12/31/2018,BIRTH_DATE,Hepatitis,50139,99.996011,BIRTH_DATE
3,46,12/17/2025,BIRTH_DATE_CPY,STD,131753,0.034914,BIRTH_DATE_CPY
4,352,ASSIGNED,CASE_INVESTIGATION_STATUS_ASSIGNED,General,283121,0.124328,CASE_INVESTIGATION_STATUS
...,...,...,...,...,...,...,...
193,29,Original interview - N/A - N/A,SYMPTOMS_INTERVIEW,STD,131753,0.022011,SYMPTOMS_INTERVIEW
194,145,12/28/2025,SYMPTOMS_PATIENT_DESCRIBED_DATE,STD,131753,0.110054,SYMPTOMS_PATIENT_DESCRIBED_DATE
195,6,NO,SYMPTOMS_PATIENT_DESCRIBED_NO,STD,131753,0.004554,SYMPTOMS_PATIENT_DESCRIBED
196,2,UNKNOWN,SYMPTOMS_PATIENT_DESCRIBED_UNKNOWN,STD,131753,0.001518,SYMPTOMS_PATIENT_DESCRIBED


In [82]:
# Group by parent_question_id and model_name to get the num of values reported
df_group = df_query.groupby(by=['parent_question_id','model_name','case_count'])['count_of_values'].sum().reset_index(name='count_of_values')

In [83]:
print(df_group)

                  parent_question_id model_name  case_count  count_of_values
0                                       General      283121               34
1                         BIRTH_DATE    General      283121           281304
2                         BIRTH_DATE  Hepatitis       50139            50137
3                         BIRTH_DATE        STD      131753           131749
4                     BIRTH_DATE_CPY        STD      131753               46
..                               ...        ...         ...              ...
115      SYMPTOMS_CLINICIAN_OBS_DATE        STD      131753              214
116                SYMPTOMS_DURATION        STD      131753               83
117               SYMPTOMS_INTERVIEW        STD      131753               29
118       SYMPTOMS_PATIENT_DESCRIBED        STD      131753              233
119  SYMPTOMS_PATIENT_DESCRIBED_DATE        STD      131753              145

[120 rows x 4 columns]


In [84]:
df_group.loc[:,'fill_rate'] = [round(c*100.0/t,3) for c,t in zip(df_group.loc[:,'count_of_values'],df_group.loc[:,'case_count'])]

In [85]:
df_group

,parent_question_id,model_name,case_count,count_of_values,fill_rate
0,,General,283121,34,0.012
1,BIRTH_DATE,General,283121,281304,99.358
2,BIRTH_DATE,Hepatitis,50139,50137,99.996
3,BIRTH_DATE,STD,131753,131749,99.997
4,BIRTH_DATE_CPY,STD,131753,46,0.035
...,...,...,...,...,...
115,SYMPTOMS_CLINICIAN_OBS_DATE,STD,131753,214,0.162
116,SYMPTOMS_DURATION,STD,131753,83,0.063
117,SYMPTOMS_INTERVIEW,STD,131753,29,0.022
118,SYMPTOMS_PATIENT_DESCRIBED,STD,131753,233,0.177


In [86]:
df_group.to_csv('output/df_group.csv',index=False)

In [73]:
df_maven

,Priority,Model,Source Field Name,Source Label,CSTE Label
0,,General,BIRTH_DATE,Birth date,Birth Date
1,,Hepatitis,BIRTH_DATE,Birth date,Birth Date
2,,STD,BIRTH_DATE,Birth date,Birth Date
3,,General,CASE_INVESTIGATION_STATUS,Case investigation status,Case Class Status Code
4,,Hepatitis,CASE_INVESTIGATION_STATUS,,Case Class Status Code
...,...,...,...,...,...
56,,Childhood Lead,RACE_OTHER_SPECIFY,,Other Race Text
57,,STD,REPORT_DATE,,Date Reported
58,,General,REPORT_TO_CDC,Report case to CDC,Immediate National Notifiable Condition
59,,Hepatitis,REPORT_TO_CDC,Report Case to CDC,Immediate National Notifiable Condition


In [55]:
df_merge = pd.merge(df_maven,df_group,left_on=['Model','Source Field Name'],right_on=['model_name','parent_question_id'],how='left')

In [56]:
df_merge = df_merge[['Priority','fill_rate','Model','Source Field Name','Source Label','CSTE Label']]

In [57]:
df_merge

,Priority,fill_rate,Model,Source Field Name,Source Label,CSTE Label
0,,99.358,General,BIRTH_DATE,Birth date,Birth Date
1,,99.996,Hepatitis,BIRTH_DATE,Birth date,Birth Date
2,,99.997,STD,BIRTH_DATE,Birth date,Birth Date
3,,0.238,General,CASE_INVESTIGATION_STATUS,Case investigation status,Case Class Status Code
4,,0.913,Hepatitis,CASE_INVESTIGATION_STATUS,,Case Class Status Code
...,...,...,...,...,...,...
56,,0.011,Childhood Lead,RACE_OTHER_SPECIFY,,Other Race Text
57,,6.844,STD,REPORT_DATE,,Date Reported
58,,0.964,General,REPORT_TO_CDC,Report case to CDC,Immediate National Notifiable Condition
59,,1.141,Hepatitis,REPORT_TO_CDC,Report Case to CDC,Immediate National Notifiable Condition


In [59]:
df_merge.to_csv('output/mvp_remapping_fill_rate.csv',index=False)